In [ ]:
import pandas as pd

df_experiments = pd.read_csv("../data/clean/df_final_experiment_clients_clean.csv")
df_final_demo = pd.read_csv("../data/clean/df_final_demo_clean.csv", sep=';', index_col=0)
df_web_data = pd.read_csv("../data/clean/df_final_merged.csv", sep=';', index_col=0)

df_final_merge = (
    df_experiments
    .merge(df_web_data, on='client_id', how='left')
    .merge(df_final_demo, on='client_id', how='left')
)

In [41]:
df_final_merge

,client_id,Variation,visitor_id,visit_id,process_step,date_time,client_tenure_yr,client_tenure_mnth,client_age,gender,num_accts,total_balance,calls_6_mnth,logons_6_mnth
0,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
1,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
2,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
3,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
4,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
449826,9895983,NaN,473024645_56027518531,498981662_93503779869_272484,step_3,2017-06-15 19:52:09,10.0,121.0,49.0,Male,2.0,97748.22,2.0,2.0
449827,9895983,NaN,473024645_56027518531,498981662_93503779869_272484,step_2,2017-06-15 19:50:37,10.0,121.0,49.0,Male,2.0,97748.22,2.0,2.0
449828,9895983,NaN,473024645_56027518531,498981662_93503779869_272484,step_1,2017-06-15 19:50:05,10.0,121.0,49.0,Male,2.0,97748.22,2.0,2.0
449829,9895983,NaN,473024645_56027518531,498981662_93503779869_272484,start,2017-06-15 19:50:00,10.0,121.0,49.0,Male,2.0,97748.22,2.0,2.0


In [42]:
print(df_experiments.shape, df_web_data.shape, df_final_demo.shape, df_final_merge.shape)

# every experiment client should still be there
print(df_experiments['client_id'].nunique(), df_final_merge['client_id'].nunique())

# clients with no web activity or no demographics
print(df_final_merge['visit_id'].isna().sum())            # adjust to a column from the web file
print(df_final_merge['client_age'].isna().sum())            # adjust to a column from the demo file

# client_id should have the same dtype in all three files
print(df_experiments['client_id'].dtype, df_web_data['client_id'].dtype, df_final_demo['client_id'].dtype)

# column names clashing between files
print([c for c in df_final_merge.columns if c.endswith(('_x', '_y'))])

(70609, 2) (755405, 5) (70609, 9) (449831, 14)
70609 70609
0
127
int64 int64 int64
[]


In [50]:
df_final_merge = df_final_merge.drop_duplicates()
print(df_final_merge.duplicated().sum())

0


In [55]:
df_final_merge = df_final_merge.rename(columns={"Variation": "variation"})

In [65]:
df_final_merge.groupby('variation')['client_id'].nunique()      # group sizes, in clients
df_final_merge.groupby('variation')['date_time'].agg(['min', 'max'])   # same period

,min,max
variation,,
Control,2017-03-15 00:19:28,2017-06-20 23:57:06
Test,2017-03-15 00:43:23,2017-06-20 23:21:23


In [57]:
# Clients without a test/control label (should be 0 or you must drop them)
print(df_final_merge['variation'].isna().sum())
print(df_final_merge['variation'].value_counts(dropna=False))

# Clients per group (not rows)
print(df_final_merge.groupby('variation')['client_id'].nunique())

126662
variation
Test       176699
Control    140536
NaN        126662
Name: count, dtype: int64
variation
Control    23532
Test       26968
Name: client_id, dtype: int64


In [59]:
visits = (df_final_merge
          .groupby(['visit_id', 'variation'])['process_step']
          .agg(lambda s: 'confirm' in set(s))
          .reset_index(name='completed'))

rate = visits.groupby('variation')['completed'].agg(['mean', 'sum', 'count'])
rate['mean'] = (rate['mean'] * 100).round(2)
rate

,mean,sum,count
variation,,,
Control,49.85,16046,32189
Test,58.52,21731,37136


In [61]:
clients = df_final_merge.drop_duplicates('client_id')
clients.groupby('variation')[['client_age', 'client_tenure_yr']].mean().round(2)

,client_age,client_tenure_yr
variation,,
Control,47.50,12.09
Test,47.17,11.98


## Main KPI: completion rate

In [64]:
visits = (df_final_merge.groupby(['visit_id', 'variation'])['process_step']
          .agg(lambda s: 'confirm' in set(s)).reset_index(name='completed'))
rate = visits.groupby('variation')['completed'].agg(['mean', 'sum', 'count'])
rate

,mean,sum,count
variation,,,
Control,0.498493,16046,32189
Test,0.585173,21731,37136


In [6]:
#df_final_merge.to_csv("../data/clean/merged_data_sets.csv", sep=";", encoding="utf-8")

In [7]:
df = df_final_merge

In [24]:
df.info()

<class 'pandas.DataFrame'>
Index: 443897 entries, 0 to 449830
Data columns (total 14 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   client_id           443897 non-null  int64  
 1   variation           317235 non-null  str    
 2   visitor_id          443897 non-null  str    
 3   visit_id            443897 non-null  str    
 4   process_step        443897 non-null  str    
 5   date_time           443897 non-null  str    
 6   client_tenure_yr    443784 non-null  float64
 7   client_tenure_mnth  443784 non-null  float64
 8   client_age          443772 non-null  float64
 9   gender              443897 non-null  str    
 10  num_accts           443784 non-null  float64
 11  total_balance       443784 non-null  float64
 12  calls_6_mnth        443784 non-null  float64
 13  logons_6_mnth       443784 non-null  float64
dtypes: float64(7), int64(1), str(6)
memory usage: 50.8 MB


In [14]:
df.duplicated().sum()

np.int64(5934)

In [16]:
df = df.drop_duplicates()
print(df.duplicated().sum())

0


In [66]:
demo = df_final_merge.copy()
print(demo.shape)
demo.info()
print(demo.isna().sum())
print(demo.duplicated(subset="client_id").sum())   # should be 0

(443897, 14)
<class 'pandas.DataFrame'>
Index: 443897 entries, 0 to 449830
Data columns (total 14 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   client_id           443897 non-null  int64  
 1   variation           317235 non-null  str    
 2   visitor_id          443897 non-null  str    
 3   visit_id            443897 non-null  str    
 4   process_step        443897 non-null  str    
 5   date_time           443897 non-null  str    
 6   client_tenure_yr    443784 non-null  float64
 7   client_tenure_mnth  443784 non-null  float64
 8   client_age          443772 non-null  float64
 9   gender              443897 non-null  str    
 10  num_accts           443784 non-null  float64
 11  total_balance       443784 non-null  float64
 12  calls_6_mnth        443784 non-null  float64
 13  logons_6_mnth       443784 non-null  float64
dtypes: float64(7), int64(1), str(6)
memory usage: 50.8 MB
client_id                  0
vari

In [17]:
df['gender'].unique()

<StringArray>
['Unspecified', 'Male', 'Female']
Length: 3, dtype: str

In [18]:
df['client_age'].value_counts()

client_age
58.0    16924
62.0    16137
52.0    16122
54.0    16047
60.0    15946
        ...  
89.0       68
94.0       61
91.0       43
93.0       27
96.0       16
Name: count, Length: 82, dtype: int64

In [19]:
df['gender'].value_counts()

gender
Unspecified    149994
Male           149867
Female         144036
Name: count, dtype: int64

In [20]:
df['total_balance'].value_counts()

total_balance
176731.72     81
24449.73      75
87107.79      74
126442.72     72
2797594.12    69
              ..
53510.23       1
48349.79       1
169121.67      1
43326.30       1
27701.79       1
Name: count, Length: 70336, dtype: int64

In [29]:
df_test = df[df['variation']=='Test']
df_test

,client_id,variation,visitor_id,visit_id,process_step,date_time,client_tenure_yr,client_tenure_mnth,client_age,gender,num_accts,total_balance,calls_6_mnth,logons_6_mnth
0,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
1,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
2,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
3,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
4,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,5.0,64.0,79.0,Unspecified,2.0,189023.86,1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321304,6334360,Test,629124187_65258232847,586047816_14599436811_389185,step_2,2017-04-11 08:45:38,4.0,53.0,66.0,Unspecified,2.0,93103.47,6.0,9.0
321305,6334360,Test,629124187_65258232847,586047816_14599436811_389185,step_1,2017-04-11 08:44:43,4.0,53.0,66.0,Unspecified,2.0,93103.47,6.0,9.0
321306,6334360,Test,629124187_65258232847,586047816_14599436811_389185,step_2,2017-04-11 08:44:30,4.0,53.0,66.0,Unspecified,2.0,93103.47,6.0,9.0
321307,6334360,Test,629124187_65258232847,586047816_14599436811_389185,step_1,2017-04-11 08:42:36,4.0,53.0,66.0,Unspecified,2.0,93103.47,6.0,9.0


In [31]:
df_control = df[df['variation']=='Control']
df_control

,client_id,variation,visitor_id,visit_id,process_step,date_time,client_tenure_yr,client_tenure_mnth,client_age,gender,num_accts,total_balance,calls_6_mnth,logons_6_mnth
20,4033851,Control,762728880_76361333336,949661017_22392791362_127391,confirm,2017-04-05 12:29:03,12.0,149.0,64.0,Male,2.0,142642.26,5.0,8.0
21,4033851,Control,762728880_76361333336,949661017_22392791362_127391,confirm,2017-04-05 12:29:01,12.0,149.0,64.0,Male,2.0,142642.26,5.0,8.0
22,4033851,Control,762728880_76361333336,949661017_22392791362_127391,confirm,2017-04-05 12:28:52,12.0,149.0,64.0,Male,2.0,142642.26,5.0,8.0
23,4033851,Control,762728880_76361333336,949661017_22392791362_127391,step_3,2017-04-05 12:26:08,12.0,149.0,64.0,Male,2.0,142642.26,5.0,8.0
24,4033851,Control,762728880_76361333336,949661017_22392791362_127391,step_2,2017-04-05 12:24:43,12.0,149.0,64.0,Male,2.0,142642.26,5.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
321265,393005,Control,238199953_33257685972,579367767_60788307935_908431,start,2017-03-29 15:25:20,15.0,191.0,52.0,Male,2.0,60344.67,1.0,4.0
321266,2908510,Control,814969699_90652851448,562606085_36368381773_92090,start,2017-05-10 22:57:17,21.0,252.0,34.0,Male,3.0,141808.05,6.0,9.0
321267,2908510,Control,814969699_90652851448,562606085_36368381773_92090,step_2,2017-05-10 22:56:31,21.0,252.0,34.0,Male,3.0,141808.05,6.0,9.0
321268,2908510,Control,814969699_90652851448,562606085_36368381773_92090,step_1,2017-05-10 22:56:23,21.0,252.0,34.0,Male,3.0,141808.05,6.0,9.0


In [33]:
result = len(df) -(len(df_test)+len(df_control))
result

126662

In [ ]:
what should be the next steps of analysis